In [ ]:
pip install -q pandas wandb torch transformers datasets peft accelerate bitsandbytes trl tqdm evaluate bert_score rouge_score nltk sacrebleu codebleu

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer,BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import gc
import os
import wandb
import evaluate
import numpy as np
import nltk

In [ ]:
from datasets import load_dataset

train_dataset = load_dataset("alenphilip/Code-Review-Assistant", split="train")
eval_dataset = load_dataset("alenphilip/Code-Review-Assistant-Eval", split="train")

print("Columns in dataset:", train_dataset.column_names)
print("\nFirst Example Sample:\n", train_dataset[0]["text"])

README.md:   0%|          | 0.00/5.16k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/8.45M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/13670 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/5.20k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/922k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1726 [00:00<?, ? examples/s]

Columns in dataset: ['text']

First Example Sample:
 <|im_start|>system
You are a helpful AI assistant specialized in code review and security analysis.<|im_end|>
<|im_start|>user
Review this Python code and provide improvements with fixed code:

```python
def get_user_by_email(email):
    query = "SELECT * FROM users WHERE email = '" + email + "'"
    cursor.execute(query)
    return cursor.fetchone()
```<|im_end|>
<|im_start|>assistant
Critical Security Issue: SQL Injection Vulnerability

The code uses string concatenation to build SQL queries, allowing attackers to inject malicious SQL code. An attacker could input something like "admin@test.com' OR '1'='1" to bypass authentication or extract sensitive data.

Fixed Code:
```python
def get_user_by_email(email: str) -> tuple:
    query = "SELECT * FROM users WHERE email = %s"
    cursor.execute(query, (email,))
    return cursor.fetchone()
```<|im_end|>



In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:

import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer
import evaluate

# Ensure memory allocations stay tightly compressed
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"
OUTPUT_DIR = "./qwen2.5-3b-sft-qlora"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

max_memory_mapping = {0: "13GiB", "cpu": "10GiB"}


model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    max_memory=max_memory_mapping,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    torch_dtype=torch.bfloat16,)

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj", "down_proj", "gate_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)

# Metrics evaluation setup
nltk.download('punkt', quiet=True)
rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")

def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim=-1)

def compute_metrics(eval_preds):
    try:
        predictions, labels = eval_preds
        predicted_ids = predictions

        predicted_ids = np.where(predicted_ids == -100, tokenizer.pad_token_id, predicted_ids)
        labels = np.where(labels == -100, tokenizer.pad_token_id, labels)

        decoded_preds = tokenizer.batch_decode(predicted_ids, skip_special_tokens=False)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=False)

        parsed_preds = []
        parsed_labels = []

        for pred, label in zip(decoded_preds, decoded_labels):
            if "<|im_start|>assistant" in pred:
                assistant_part = pred.split("<|im_start|>assistant")[-1]
                if "<|im_end|>" in assistant_part:
                    assistant_response = assistant_part.split("<|im_end|>")[0].strip()
                    parsed_preds.append(assistant_response)
                else:
                    parsed_preds.append(assistant_part.strip())
            else:
                parsed_preds.append(pred.strip())

            if "<|im_start|>assistant" in label:
                assistant_part = label.split("<|im_start|>assistant")[-1]
                if "<|im_end|>" in assistant_part:
                    assistant_response = assistant_part.split("<|im_end|>")[0].strip()
                    parsed_labels.append(assistant_response)
                else:
                    parsed_labels.append(assistant_part.strip())
            else:
                parsed_labels.append(label.strip())

        parsed_preds = [p for p in parsed_preds if p.strip()]
        parsed_labels = [l for l in parsed_labels if l.strip()]

        if not parsed_preds or not parsed_labels:
            return {"rougeL": 0.0, "bleu": 0.0}

        rouge_results = rouge_metric.compute(predictions=parsed_preds, references=parsed_labels, use_stemmer=True)
        bleu_results = bleu_metric.compute(predictions=parsed_preds, references=[[l] for l in parsed_labels])

        return {
            "rougeL": rouge_results["rougeL"],
            "bleu": bleu_results["bleu"],
        }
    except Exception as e:
        print(f"Metrics parsing fallback triggered. Error details: {e}")
        return {"rougeL": 0.0, "bleu": 0.0}

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    do_train=True,
    do_eval=True,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    max_length=512,

    fp16=False,
    bf16=True,
    prediction_loss_only=False,

    eval_strategy="steps",
    eval_steps=100,
    eval_accumulation_steps=10,

    learning_rate=3e-4,
    num_train_epochs=1,
    max_grad_norm=0.3,
    optim="paged_adamw_8bit",

    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,

    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.05,

    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,

    dataset_text_field="text",
    remove_unused_columns=False,
    gradient_checkpointing=True,
    dataloader_pin_memory=False,

    push_to_hub=True,
    hub_model_id="rose00009/code-reviewer",
    hub_strategy="all_checkpoints",
    hub_always_push=True,

    report_to="wandb"
)

# 7. Start Training
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
)

print("Starting training securely on L4 GPU...")
print(f"Training on {len(train_dataset)} examples")
print(f"Evaluating on {len(eval_dataset)} examples every {training_args.eval_steps} steps")
print(f"Best model will be selected based on {training_args.metric_for_best_model}")

trainer.train()

print("\n" + "="*70)
print("Training completed!")
print("="*70)

trainer.save_model("./final_code_reviewer_model")
trainer.push_to_hub("Training complete!")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Starting training securely on L4 GPU...
Training on 13670 examples
Evaluating on 1726 examples every 100 steps
Best model will be selected based on rougeL


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: roseebby06 (roseebby06-national-institute-of-technology-karnataka) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss,Rougel,Bleu,Entropy,Num Tokens,Mean Token Accuracy
100,0.738763,0.672577,0.728695,0.639408,0.672156,1140000.000000,0.813561
200,0.682395,0.649879,0.736263,0.646458,0.624152,2275606.000000,0.818851
300,0.661899,0.638337,0.740290,0.649857,0.622786,3412058.000000,0.821619
400,0.684139,0.636884,0.740767,0.650416,0.612777,4545921.000000,0.821852
428,0.673318,0.636878,0.740731,0.650353,0.612733,4855797.000000,0.821758



Training completed!


HfHubHTTPError: (Request ID: Root=1-6a284106-6326451b5c94552c08938fbc;b4feb664-b816-4477-80c2-e966141f559e)

403 Forbidden: Forbidden: you must use a write token to upload to a repository..
Cannot access content at: https://huggingface.co/api/models/rose00009/code-reviewer/preupload/main.
Make sure your token has the correct permissions.

In [ ]:
# Upgrade torchao to the version required by PEFT
!pip install -U torchao

# Force upgrade peft and transformers just to keep everything perfectly in sync
!pip install -U peft transformers trl bitsandbytes

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel


BASE_MODEL_ID = "Qwen/Qwen2.5-Coder-3B-Instruct"
HF_ADAPTER_ID = "rose00009/code-reviewer"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)

model = PeftModel.from_pretrained(base_model, HF_ADAPTER_ID)

print("The code-reviewer model is fully assembled and ready to test.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/74.9M [00:00<?, ?B/s]

The code-reviewer model is fully assembled and ready to test.


In [ ]:
messages = [
    {"role": "system", "content": "You are an expert code reviewer."},
    {"role": "user", "content": "Review this Python function:\n\ndef process_data(vals):\n    for i in range(len(vals)):\n        print(vals[i])"}
]

# Format using Qwen's specific template syntax
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors="pt").to("cuda")

# Generate response ids
with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.4,
        do_sample=True
    )

generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, generated_ids)]
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("\n--- Code Review Output ---")
print(response)


--- Code Review Output ---
Issue Identification: Readability and Performance - Using `range(len(...))` for Indexing

Problem Analysis: The code uses `range(len(vals))` to iterate over the indices of a list. This is a common pattern but can be slightly less readable than using `enumerate`, which provides both the index and the value at the same time. Furthermore, if the list is modified during iteration, `range(len(...))` will not reflect the changes, leading to unexpected behavior.

Fixed Code:
```python
from typing import List, Any

def process_data(vals: List[Any]) -> None:
    for i, val in enumerate(vals):
        print(val)
```


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from huggingface_hub import notebook_login

print("Log in using your Hugging Face WRITE token:")
notebook_login()

In [ ]:


BASE_MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"
HF_ADAPTER_ID = "rose00009/code-reviewer"
TARGET_REPO = "rose00009/Code_Review_Assistant_Model1"

# Load the unquantized base model
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

# attach custom adapter files over the base model structures
model_with_adapter = PeftModel.from_pretrained(base_model, HF_ADAPTER_ID)

merged_model = model_with_adapter.merge_and_unload()

merged_model.push_to_hub(TARGET_REPO, max_shard_size="2GB")

tokenizer.push_to_hub(TARGET_REPO)

print("Your fully merged, custom-trained model is officially live on your profile.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

...

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp_udt95ys/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Your fully merged, custom-trained model is officially live on your profile.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"
HF_ADAPTER_ID = "rose00009/code-reviewer"
LOCAL_OUTPUT_DIR = "./my_local_code_reviewer"

# Load the unquantized base model
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

model_with_adapter = PeftModel.from_pretrained(base_model, HF_ADAPTER_ID)

merged_model = model_with_adapter.merge_and_unload()

merged_model.save_pretrained(
    LOCAL_OUTPUT_DIR,
    max_shard_size="2GB",
    safe_serialization=False  # forces the traditional .bin format
)

tokenizer.save_pretrained(LOCAL_OUTPUT_DIR)

print("Your fully merged model is saved locally.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

adapter_config.json:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/74.9M [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/4 [00:00<?, ?it/s]

Your fully merged model is saved locally.
